# 01 — Exploratory Data Analysis

Visualize OASIS-1 and OASIS-2 features stratified by dementia status.

**Requirements:** FR-01 through FR-05 (PRD)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
from src.data_loader import load_and_merge
from src.preprocessing import create_target
from src.utils import SEED, data_path, results_path, set_seed

set_seed()
sns.set_theme(style='whitegrid', palette='colorblind')
FIG_DIR = results_path('figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load raw datasets
df1, df2, merged = load_and_merge()
print(f'OASIS-1: {df1.shape} | OASIS-2: {df2.shape} | Merged (baseline): {merged.shape}')
merged.head()

In [ ]:
# FR-05: Class imbalance distribution
merged_labeled = create_target(merged.dropna(subset=['CDR']))
class_counts = merged_labeled['target'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind='bar', ax=ax, color=sns.color_palette('colorblind', 2))
ax.set_title('Class Distribution (Demented vs Non-Demented)')
ax.set_xlabel('Target (0=Non-demented, 1=Demented)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig01_class_distribution.png', dpi=300)
plt.show()

In [ ]:
# FR-04: Missing data patterns
missing = merged.isnull().mean().sort_values(ascending=False)
missing_pct = (missing * 100).round(1)
print(missing_pct[missing_pct > 0])

In [ ]:
# FR-02: Correlation heatmap for numerical features
num_cols = ['Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF']
num_cols = [c for c in num_cols if c in merged.columns]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(merged[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig02_correlation_heatmap.png', dpi=300)
plt.show()

In [ ]:
# FR-03: MMSE vs nWBV scatter colored by dementia status
plot_df = merged_labeled.dropna(subset=['MMSE', 'nWBV'])

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=plot_df, x='MMSE', y='nWBV', hue='target', palette='colorblind', ax=ax)
ax.set_title('MMSE vs Normalized Whole Brain Volume')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig03_mmse_nwbv_scatter.png', dpi=300)
plt.show()

In [ ]:
# FR-01: Feature distributions stratified by dementia status
features = ['Age', 'MMSE', 'nWBV', 'EDUC']
features = [f for f in features if f in merged_labeled.columns]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.ravel(), features):
    sns.histplot(data=merged_labeled, x=feat, hue='target', kde=True, ax=ax, palette='colorblind')
    ax.set_title(f'{feat} by Dementia Status')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig01_feature_distributions.png', dpi=300)
plt.show()